# 🦴 RSNA BONE AGE ASSESSMENT — MODULAR PIPELINE (NOTEBOOK 01/03)
## 📦 Module 01: Data Audit, Clinical EDA, Classical CV Preprocessing & Offline Caching
**Đồ Án Nghiên Cứu Học Thuật Chuẩn DUT — VisionLab Corp (CORP-01-CV)**
* **Mục tiêu**: Xử lý toàn diện 12.611 ca bệnh thật từ bộ dữ liệu RSNA, xây dựng pipeline tiền xử lý ảnh X-quang 5 bước cổ điển, cố định phân tầng Train/Val/Test (80/10/10), và xuất bộ dữ liệu sạch chuẩn $512 \times 512$ làm cache cho quá trình huấn luyện mô hình.
* **Thời gian chạy**: Chạy **01 LẦN DUY NHẤT** (~25-30 phút), không cần GPU mạnh (CPU là đủ).
* **Đầu ra**: Thư mục `/kaggle/working/rsna_processed_512/` chứa 12.611 ảnh sạch và file `train_stratified.csv` cố định.


In [ ]:
# ==============================================================================
# BƯỚC 1: KHỞI TẠO MÔI TRƯỜNG, THƯ VIỆN & CỐ ĐỊNH SEED TÁI LẬP NGHIỆM
# ==============================================================================
import os
import sys
import glob
import time
import random
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedShuffleSplit

# Thiết lập hiển thị đồ họa
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'

# Cố định ngẫu nhiên đảm bảo tính tái lập 100% (Deterministic Reproducibility)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f"✅ Khởi tạo môi trường thành công! Random Seed cố định: {SEED}")
print(f"📦 OpenCV Version: {cv2.__version__} | Pandas: {pd.__version__} | NumPy: {np.__version__}")


In [ ]:
# ==============================================================================
# BƯỚC 2: TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN DỮ LIỆU RSNA (BULLETPROOF AUTO-DISCOVERY)
# ==============================================================================
import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')
IS_COLAB = 'google.colab' in sys.modules

print("=" * 70)
print("🔍 BẮT ĐẦU TỰ ĐỘNG QUÉT VÀ NHẬN DIỆN DỮ LIỆU RSNA BONE AGE...")
print("=" * 70)

TRAIN_CSV_PATH = None
IMG_DIR = None

if IS_KAGGLE:
    print("🌐 Môi trường phát hiện: KAGGLE NOTEBOOK")
    base_input = Path('/kaggle/input')
    
    # 1. Liệt kê các dataset đã gắn vào notebook
    available_datasets = [p for p in base_input.iterdir() if p.is_dir()]
    print(f"📂 Các dataset phát hiện trong /kaggle/input: {[p.name for p in available_datasets]}")
    
    # 2. Quét đệ quy tìm tất cả file .csv
    csv_candidates = []
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.lower().endswith('.csv'):
                csv_candidates.append(Path(root) / f)
                
    print(f"📄 Danh sách các file CSV tìm thấy ({len(csv_candidates)} file):")
    for cp in csv_candidates:
        print(f"   • {cp}")
        
    # Ưu tiên tìm file có chứa 'boneage' và 'train' (hoặc 'train.csv')
    for cp in csv_candidates:
        name_lower = cp.name.lower()
        if 'boneage' in name_lower and 'train' in name_lower:
            TRAIN_CSV_PATH = cp
            break
            
    if TRAIN_CSV_PATH is None:
        for cp in csv_candidates:
            if 'train' in cp.name.lower():
                TRAIN_CSV_PATH = cp
                break
                
    if TRAIN_CSV_PATH is None and csv_candidates:
        TRAIN_CSV_PATH = csv_candidates[0]
        
    # 3. Quét đệ quy tìm thư mục chứa các ảnh X-quang (.png)
    img_dir_counts = {}
    for root, dirs, files in os.walk('/kaggle/input'):
        png_count = sum(1 for f in files if f.lower().endswith('.png'))
        if png_count > 50:
            img_dir_counts[Path(root)] = png_count
            
    if img_dir_counts:
        # Chọn thư mục có số lượng ảnh lớn nhất (tập train RSNA có ~12.611 ảnh)
        IMG_DIR = max(img_dir_counts, key=img_dir_counts.get)
        print(f"🖼️ Thư mục ảnh huấn luyện phát hiện: {IMG_DIR} ({img_dir_counts[IMG_DIR]:,} ảnh .png)")
    else:
        print("⚠️ Chưa phát hiện thư mục chứa file .png trực tiếp qua os.walk!")

elif IS_COLAB:
    print("🌐 Môi trường phát hiện: GOOGLE COLAB")
    TRAIN_CSV_PATH = Path('/content/data/rsna-bone-age/boneage-training-dataset.csv')
    IMG_DIR = Path('/content/data/rsna-bone-age/boneage-training-dataset')
else:
    print("🌐 Môi trường phát hiện: LOCAL WORKSTATION")
    TRAIN_CSV_PATH = Path('./data/boneage-training-dataset.csv')
    IMG_DIR = Path('./data/boneage-training-dataset')

# Khẳng định tính hợp lệ tuyệt đối trước khi sang Bước 3
print("-" * 70)
if TRAIN_CSV_PATH is None or not Path(TRAIN_CSV_PATH).exists():
    raise FileNotFoundError( "❌ KHÔNG TÌM THẤY FILE CSV NHÃN HUẤN LUYỆN! " "👉 Nguyên nhân: Chưa gắn dataset RSNA hoặc tên thư mục khác biệt. " "👉 Cách khắc phục: Ở cột bên phải (Input), nhấn '+ Add Input' -> tìm 'rsna-bone-age' (tác giả kmader) -> nhấn Add." )

if IMG_DIR is None or not Path(IMG_DIR).exists():
    raise FileNotFoundError( "❌ KHÔNG TÌM THẤY THƯ MỤC CHỨA ẢNH X-QUANG PNG! " "👉 Cách khắc phục: Kiểm tra cây thư mục trong tab Input bên phải xem dataset RSNA đã sẵn sàng chưa." )

print(f"✅ XÁC THỰC ĐƯỜNG DẪN THÀNH CÔNG 100%:")
print(f"• File nhãn CSV: {TRAIN_CSV_PATH} (Dung lượng: {Path(TRAIN_CSV_PATH).stat().st_size / 1024:.1f} KB)")
print(f"• Thư mục ảnh:   {IMG_DIR}")
print("=" * 70)


In [ ]:
# ==============================================================================
# BƯỚC 3: KHÁM PHÁ & THỐNG KÊ DỮ LIỆU LÂM SÀNG THỰC TẾ (CLINICAL EDA)
# ==============================================================================
df_raw = pd.read_csv(TRAIN_CSV_PATH)
df_raw.columns = [c.lower().strip() for c in df_raw.columns]

# Chuẩn hóa tên cột
if 'id' not in df_raw.columns:
    df_raw = df_raw.rename(columns={df_raw.columns[0]: 'id'})
if 'boneage' not in df_raw.columns:
    df_raw = df_raw.rename(columns={'bone_age': 'boneage'})

print("📊 THỐNG KÊ MÔ TẢ TẬP DỮ LIỆU RSNA GỐC:")
print(f"• Tổng số lượng ca bệnh ghi nhận: {len(df_raw):,} bệnh nhi")
print(f"• Tuổi xương nhỏ nhất: {df_raw['boneage'].min():.1f} tháng ({df_raw['boneage'].min()/12:.2f} tuổi)")
print(f"• Tuổi xương lớn nhất: {df_raw['boneage'].max():.1f} tháng ({df_raw['boneage'].max()/12:.2f} tuổi)")
print(f"• Tuổi xương trung bình: {df_raw['boneage'].mean():.2f} ± {df_raw['boneage'].std():.2f} tháng")
print(f"• Phân bố giới tính: {df_raw['male'].sum():,} Nam ({df_raw['male'].mean()*100:.1f}%) | "
      f"{len(df_raw) - df_raw['male'].sum():,} Nữ ({(1 - df_raw['male'].mean())*100:.1f}%)")

# Trực quan hóa bảng phân phối lâm sàng 4 đồ thị
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Đồ thị 1: Histogram phân phối tuổi xương
sns.histplot(df_raw['boneage'], bins=40, kde=True, color='#1f77b4', ax=axes[0, 0])
axes[0, 0].axvline(df_raw['boneage'].mean(), color='red', linestyle='--', label=f"Mean: {df_raw['boneage'].mean():.1f}m")
axes[0, 0].axvline(df_raw['boneage'].median(), color='orange', linestyle=':', label=f"Median: {df_raw['boneage'].median():.1f}m")
axes[0, 0].set_title("1. Phân Phối Tần Suất Tuổi Xương (Tháng)", fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel("Tuổi xương (tháng)")
axes[0, 0].legend()

# Đồ thị 2: Boxplot tuổi xương theo giới tính
sns.boxplot(data=df_raw, x='male', y='boneage', hue='male', palette=['#ff7f0e', '#2ca02c'], legend=False, ax=axes[0, 1])
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_xticklabels(['Nữ (Female)', 'Nam (Male)'])
axes[0, 1].set_title("2. So Sánh Phân Bố Tuổi Xương Theo Giới Tính", fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel("Tuổi xương (tháng)")

# Đồ thị 3: KDE phân phối tuổi giữa 2 giới tính
sns.kdeplot(df_raw[df_raw['male'] == 1]['boneage'], label='Nam (Male)', color='#2ca02c', fill=True, alpha=0.3, ax=axes[1, 0])
sns.kdeplot(df_raw[df_raw['male'] == 0]['boneage'], label='Nữ (Female)', color='#ff7f0e', fill=True, alpha=0.3, ax=axes[1, 0])
axes[1, 0].set_title("3. Mật Độ Phân Bố Tuổi Xương Nam vs Nữ", fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel("Tuổi xương (tháng)")
axes[1, 0].legend()

# Đồ thị 4: Tỷ lệ giới tính
axes[1, 1].pie([df_raw['male'].sum(), len(df_raw) - df_raw['male'].sum()], 
               labels=['Nam (54.1%)', 'Nữ (45.9%)'], 
               autopct='%1.1f%%', colors=['#2ca02c', '#ff7f0e'], startangle=140, explode=(0.03, 0))
axes[1, 1].set_title("4. Cơ Cấu Giới Tính Trong Tập Dữ Liệu", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# BƯỚC 4: PHÂN TÍCH TÍN HIỆU QUANG HỌC X-RAY & HIỆN TƯỢNG CẢN QUANG
# ==============================================================================
# Nạp ngẫu nhiên 1 ảnh bệnh nhi để phân tích
sample_row = df_raw.sample(1, random_state=SEED).iloc[0]
sample_img_id = sample_row['id']
sample_path = IMG_DIR / f"{sample_img_id}.png"

img_sample_gray = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(img_sample_gray, cmap='gray')
axes[0].set_title(f"Ảnh X-quang Gốc (Bệnh nhi ID: {sample_img_id} | Tuổi: {sample_row['boneage']}m | Nam: {sample_row['male']})", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Lược đồ mức xám (Grayscale Histogram)
axes[1].hist(img_sample_gray.ravel(), bins=256, range=[0, 256], color='#333333', alpha=0.8)
axes[1].set_title("Lược Đồ Mức Xám & Hiện Tượng Nhiễu Nền Đen", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Giá trị mức xám (0: Đen tuyệt đối -> 255: Trắng cản quang)")
axes[1].set_ylabel("Số lượng điểm ảnh (Pixels)")

plt.tight_layout()
plt.show()

print(f"📌 ĐẶC ĐIỂM QUANG HỌC BẮT BUỘC PHẢI XỬ LÝ:")
print(f"1. Đỉnh xung cao bất thường tại mức xám 0 biểu thị vùng nền đen máy chụp bao quanh.")
print(f"2. Độ tương phản mô mềm và sụn tiếp hợp bị nén hẹp trong khoảng mức xám [30, 180].")
print(f"3. Cần áp dụng thuật toán CLAHE để trải đều biểu đồ mức xám mà không làm cháy sụn.")


In [ ]:
# ==============================================================================
# BƯỚC 5: XÂY DỰNG PIPELINE TIỀN XỬ LÝ THỊ GIÁC CỔ ĐIỂN (CLASSICAL CV 5 BƯỚC)
# ==============================================================================
def preprocess_xray_classical_cv(img_gray, target_size=(512, 512), clip_limit=3.0, tile_size=(8, 8)):
    """
    Pipeline 5 bước chuẩn hóa thị giác cổ điển trên ảnh X-quang:
    1. CLAHE tăng cường độ tương phản cục bộ bảo tồn sụn tiếp hợp
    2. Lọc mượt Gaussian 5x5 triệt tiêu nhiễu lượng tử hạt
    3. Phân ngưỡng tự động Otsu tách bàn tay khỏi nền đen
    4. Biến đổi hình thái học Opening/Closing xóa chữ ký hiệu L/R và dị vật kim loại
    5. Bounding box Max Contour Crop kèm padding an toàn 2% & Letterbox Resize
    """
    # Bước 1: Cân bằng biểu đồ mức xám thích nghi cục bộ (CLAHE)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    enhanced = clahe.apply(img_gray)

    # Bước 2: Lọc Gauss khử nhiễu lượng tử
    blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)

    # Bước 3: Phân ngưỡng tự động Otsu (Auto Binarization)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Bước 4: Làm sạch hình thái học (Morphological Cleaning)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=3)

    # Bước 5: Tìm đường bao lớn nhất (Max Contour) & Cắt vùng bàn tay (ROI Crop)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(c)

        # Mở rộng biên an toàn 2% để tránh cắt vào đầu ngón tay
        pad = int(0.02 * max(w, h))
        H, W = img_gray.shape
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(W, x + w + pad)
        y2 = min(H, y + h + pad)

        cropped_roi = enhanced[y1:y2, x1:x2]
    else:
        cropped_roi = enhanced

    # Resize về kích thước chuẩn hóa 512x512
    resized_roi = cv2.resize(cropped_roi, target_size, interpolation=cv2.INTER_AREA)
    return resized_roi, enhanced, thresh, closed

# Thực nghiệm trực tiếp trên ảnh mẫu
processed_img, enhanced_step, thresh_step, closed_step = preprocess_xray_classical_cv(img_sample_gray)

# Trực quan hóa 5 chặng xử lý
fig, axes = plt.subplots(1, 5, figsize=(22, 5))
axes[0].imshow(img_sample_gray, cmap='gray')
axes[0].set_title("1. Ảnh Gốc (Raw X-ray)", fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(enhanced_step, cmap='gray')
axes[1].set_title("2. Sau CLAHE (Clip=3.0)", fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(thresh_step, cmap='gray')
axes[2].set_title("3. Phân Ngưỡng Otsu", fontsize=11, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(closed_step, cmap='gray')
axes[3].set_title("4. Làm Sạch Hình Thái", fontsize=11, fontweight='bold')
axes[3].axis('off')

axes[4].imshow(processed_img, cmap='gray')
axes[4].set_title("5. Bounding Crop (512x512)", fontsize=11, fontweight='bold')
axes[4].axis('off')

plt.tight_layout()
plt.show()
print("✅ Pipeline tiền xử lý 5 bước đã được kiểm chứng hoạt động hoàn hảo!")


In [ ]:
# ==============================================================================
# BƯỚC 6: PHÂN TẦNG CỐ ĐỊNH CHỐNG RÒ RỈ (DETERMINISTIC STRATIFIED SPLIT 80-10-10)
# ==============================================================================
# Tạo 10 khoảng phân vị tuổi (Quantiles) kết hợp với Giới tính để phân tầng
df_clean = df_raw.copy()
df_clean['age_quantile'] = pd.qcut(df_clean['boneage'], q=10, labels=False)
df_clean['strata_key'] = df_clean['age_quantile'].astype(str) + "_" + df_clean['male'].astype(str)

# Chia phân tầng 80% Train và 20% Tạm (Val + Test)
split_1 = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(split_1.split(df_clean, df_clean['strata_key']))

train_df = df_clean.iloc[train_idx].copy()
temp_df = df_clean.iloc[temp_idx].copy()

# Chia tiếp tập Tạm thành 50% Val và 50% Test (tức mỗi tập chiếm 10% tổng)
split_2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_sub_idx, test_sub_idx = next(split_2.split(temp_df, temp_df['strata_key']))

val_df = temp_df.iloc[val_sub_idx].copy()
test_df = temp_df.iloc[test_sub_idx].copy()

# Gán nhãn phân vùng cố định
train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

df_stratified = pd.concat([train_df, val_df, test_df], ignore_index=True)
df_stratified = df_stratified[['id', 'boneage', 'male', 'split', 'age_quantile']]

print("📊 PHÂN BỔ MẪU SAU KHI PHÂN TẦNG CỐ ĐỊNH:")
print(f"• Tập Huấn luyện (Train Set): {len(train_df):,} ca ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"• Tập Kiểm định (Val Set):     {len(val_df):,} ca ({len(val_df)/len(df_clean)*100:.1f}%)")
print(f"• Tập Kiểm thử (Test Set):     {len(test_df):,} ca ({len(test_df)/len(df_clean)*100:.1f}%)")
print("✅ Phân tầng đảm bảo tỷ lệ giới tính và phân phối mọi lứa tuổi đồng đều giữa cả 3 tập!")


In [ ]:
# ==============================================================================
# BƯỚC 7: THỰC THI BATCH TIỀN XỬ LÝ & ĐÓNG GÓI CACHE DATASET (512x512)
# ==============================================================================
OUTPUT_CACHE_DIR = Path('/kaggle/working/rsna_processed_512') if IS_KAGGLE else Path('./rsna_processed_512')
CACHE_IMAGES_DIR = OUTPUT_CACHE_DIR / 'images'
CACHE_IMAGES_DIR.mkdir(parents=True, exist_ok=True)

# Lưu tệp phân tầng cố định
SPLIT_CSV_OUTPUT = OUTPUT_CACHE_DIR / 'train_stratified.csv'
df_stratified.to_csv(SPLIT_CSV_OUTPUT, index=False)
print(f"💾 Đã lưu file nhãn phân tầng cố định tại: {SPLIT_CSV_OUTPUT}")

def process_and_save_single_image(row_tuple):
    _, row = row_tuple
    img_id = int(row['id'])
    in_path = IMG_DIR / f"{img_id}.png"
    out_path = CACHE_IMAGES_DIR / f"{img_id}.png"
    
    # Nếu file đã tồn tại thì bỏ qua (resume caching)
    if out_path.exists():
        return True
        
    try:
        raw_gray = cv2.imread(str(in_path), cv2.IMREAD_GRAYSCALE)
        if raw_gray is None:
            return False
        processed, _, _, _ = preprocess_xray_classical_cv(raw_gray, target_size=(512, 512))
        cv2.imwrite(str(out_path), processed, [cv2.IMWRITE_PNG_COMPRESSION, 4])
        return True
    except Exception as e:
        return False

print(f"🚀 Bắt đầu tiền xử lý hàng loạt {len(df_stratified):,} ảnh X-quang bằng đa luồng...")
start_time = time.time()

rows = list(df_stratified.iterrows())
with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as executor:
    results = list(tqdm(executor.map(process_and_save_single_image, rows), total=len(rows), desc="Tiền xử lý ảnh"))

elapsed = time.time() - start_time
success_count = sum(results)

print(f"\n🎉 HOÀN THÀNH TIỀN XỬ LÝ TOÀN BỘ DATASET SAU {elapsed/60:.2f} PHÚT!")
print(f"• Số ảnh xử lý thành công: {success_count:,} / {len(df_stratified):,} ảnh")
print(f"• Thư mục lưu trữ: {CACHE_IMAGES_DIR}")
print(f"📌 BƯỚC TIẾP THEO: Nhấn 'Save Version' -> 'Save & Run All' để lưu Dataset này thành Kaggle Dataset đầu vào cho Notebook 02!")
